# Your own rules through the referee

**Author:** Franz Triebe · **Date:** 2026-10-11 · **Status:** template, run with an example rule

**Question.** Does a new trading idea survive the referee once it is judged together with everything already tried on the same data? This notebook is the one command for that: write a rule in Section 3, run all cells, read the verdict. Every rule you run is counted in the permanent trial ledger.

**Data.** SPY daily adjusted closes from Yahoo Finance and the Ken French daily T bill rate, as in notebooks 03 and 04.

> **Key findings**
> 1. The example rule `my_band200` (a 200 day average with a 2% buffer) does what it was built for: turnover falls from 6.6 to 1.5 a year compared with the plain 200 day rule, and the Sharpe ratio rises from 0.58 to **0.66**, rank 9 of 257.
> 2. It still does not survive the referee: its deflated Sharpe ratio is **0.52**, because 257 tries on this data set a hurdle of 0.65. The overall winner and the verdict (not distinguishable from luck) are unchanged.
> 3. The ledger now counts **257** distinct trials on this data. The example idea came from looking at the same data (the whipsaws in notebook 03), which is exactly why it has to count. Your first own rule will raise the count to 258.

**Contents**
1. [Setup](#1.-Setup)
2. [Data](#2.-Data)
3. [Your rules](#3.-Your-rules)
4. [Results](#4.-Results)
5. [Conclusions](#5.-Conclusions)
6. [Appendix: environment](#6.-Appendix:-environment)

## 1. Setup

In [1]:
import platform
from importlib.metadata import version
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import quantlab
from IPython.display import Markdown, display
from quantlab import backtest, data, pipeline, plotting, stats, strategies, validation

In [2]:
TICKER = "SPY"  # Yahoo symbol; adjusted closes make it a total return series
START = "1993-01-01"  # first date to load
LAG = 1  # trading days between a decision at the close and the first return it earns
COST_BPS = 5.0  # cost per unit of one way turnover, in basis points
INCLUDE_FAMILY = True  # judge your rules together with the 256 trials of notebook 03
LOOKAHEAD_CUTS = [
    0.3,
    0.6,
    0.9,
]  # where (share of the sample) the look ahead check cuts
ANN_FACTOR = 252  # trading days per year
PROJECT = "p01"  # project name in the trial ledger
REFRESH = False  # True downloads fresh data instead of using the cache in data/
OUTPUT_DIR = Path(quantlab.__file__).resolve().parents[1] / "output" / "p01"

In [3]:
plotting.apply_style()
pd.options.display.float_format = "{:,.4f}".format
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data

In [4]:
returns = data.load_yahoo(TICKER, start=START, refresh=REFRESH)
rf = data.load_rf_daily(refresh=REFRESH).reindex(returns.index).ffill()
prices = (1 + returns).cumprod()
data_id = pipeline.make_data_id(TICKER, returns)
n_before = validation.ledger_count(PROJECT, data_id)
assert not rf.isna().any()
print(f"Ledger before this run: {n_before} distinct trials for {data_id}")

Ledger before this run: 257 distinct trials for SPY 1993-02-01 2026-10-09


## 3. Your rules

Write each idea as a function `rule(prices) -> weights`:

* `prices` is the SPY total return index (a pandas Series of daily levels).
* Return the **target weight as of each close**, between $-1$ (fully short) and $+1$ (fully long), NaN while the rule warms up.
* Use only prices up to and including that day. **Never shift**: the backtest applies the one day lag.
* Add the function to `MY_RULES` with a name starting with `my_`.

The example below is a placeholder: replace it with your own idea (it stays counted in the ledger, as every trial does).

**Example idea: a 200 day average with a buffer.** Notebook 03 showed the 200 day rule switching in and out nine times in late 2007. A band of 2% around the average should cut those whipsaws: buy only when the price is 2% above the average, sell only when it is 2% below, and in between keep the last decision.

In [5]:
def my_band200(prices):
    """Long when the price is 2% above its 200 day average, cash when 2% below, else hold."""
    average = prices.rolling(200).mean()
    decision = pd.Series(np.nan, index=prices.index)
    decision[prices > 1.02 * average] = 1.0
    decision[prices < 0.98 * average] = 0.0
    return decision.ffill().where(average.notna())


MY_RULES = {"my_band200": my_band200}

Before anything is backtested, every rule must pass the same look ahead test as the 256 trials: halve all prices after a cut date and check that no weight up to that date changes. A rule that fails is stopped here.

In [6]:
def passes_lookahead_check(rule):
    """True if no weight up to a cut date changes when all later prices are changed."""
    base = rule(prices)
    for share in LOOKAHEAD_CUTS:
        cut = prices.index[int(len(prices) * share)]
        changed = prices.where(prices.index <= cut, prices * 0.5)
        if not base.loc[:cut].equals(rule(changed).loc[:cut]):
            return False
    return True


my_weights = {}
for name, rule in MY_RULES.items():
    weights = rule(prices).rename(name)
    assert name.startswith("my_"), f"{name}: rule names must start with my_"
    assert weights.dropna().between(-1, 1).all(), (
        f"{name}: weights must lie between -1 and 1"
    )
    assert passes_lookahead_check(rule), (
        f"{name}: uses future prices, fix it before backtesting"
    )
    my_weights[name] = weights
print(f"{len(my_weights)} rule(s) passed the checks: {', '.join(my_weights)}")

1 rule(s) passed the checks: my_band200


## 4. Results

### 4.1 The referee on the whole family plus your rules

In [7]:
all_trials = (strategies.build_trials(prices) if INCLUDE_FAMILY else {}) | my_weights
run = pipeline.run_referee(
    all_trials,
    returns,
    project=PROJECT,
    data_label=TICKER,
    rf=rf,
    lag=LAG,
    cost_bps=COST_BPS,
)
report = run["report"]
print(
    f"Ledger after this run: {run['n_ledger']} distinct trials (before: {n_before}); this run had {len(all_trials)}"
)
display(Markdown(validation.report_to_markdown(report)))

Ledger after this run: 257 distinct trials (before: 257); this run had 257


**Selected trial:** `x50_250_lf_vt_w` · 1994-04-11 to 2026-10-09 (32.5 years, T = 8,180)

| Measure | Value | Meaning |
| --- | --- | --- |
| Sharpe ratio (annualized) | 0.68 | before any correction |
| t statistic, iid / Newey West (10 lags) | 3.90 / 4.36 | above 2 is the usual bar, above 3 after a search |
| Skewness / kurtosis | -0.51 / 8.8 | normal is 0 / 3 |
| PSR(0) | 1.000 | probability the true Sharpe is above 0, ignoring the search |
| Trials: this run / ledger / used | 257 / 257 / 257 | |
| Effective trials: avg corr / eigen | 105.1 / 2.5 | independent bets among the trials |
| Hurdle SR0 (annualized): raw N / avg corr / eigen | 0.65 / 0.58 / 0.16 | expected best Sharpe of that many useless trials |
| DSR: raw N / avg corr / eigen | 0.577 / 0.719 / 0.998 | probability the true Sharpe beats the hurdle |
| MinTRL in years vs 0 / vs SR0 (actual 32.5) | 5.9 / 2300.7 | track record needed at 95% confidence |
| Haircut Sharpe, Bonferroni | 0.39 (42% haircut) | Sharpe left after correcting for 257 tests |
| Haircut Sharpe, Holm | 0.39 (42% haircut) | Sharpe left after correcting for 257 tests |
| Haircut Sharpe, BHY | 0.42 (38% haircut) | Sharpe left after correcting for 257 tests |
| PBO (CSCV, 16 blocks, 12,870 splits) | 0.10 | share of splits where the IS winner is at or below the OOS median |
| P(OOS loss) / degradation slope | 0.00 / -0.90 | slope 1 means IS performance carries over fully |

**Verdict: Not distinguishable from luck.** Strong needs DSR at least 0.95 and PBO at most 0.2; Weak needs DSR at least 0.8 and PBO at most 0.5. These thresholds are conventions, not laws.

**Reading the table.** The overall winner is still `x50_250_lf_vt_w`. One extra trial barely moves the hurdle (0.65) or the DSR (0.577 instead of 0.579), but the track record needed to beat the hurdle grows from about 2,200 to 2,300 years. One more idea costs little; hundreds of them add up, and every new rule has to beat the best of everything tried before it.

> **Note:** the example rule was first run while this notebook was being built, so the ledger already counted it before this run (257 before and after). That earlier run was real, so it stays in the ledger.

### 4.2 Your rules against the hurdle

Each of your rules gets its own probabilistic and deflated Sharpe ratio: against 0, and against the luck hurdle for everything ever tried on this data.

In [8]:
sharpes = run["excess"].mean() / run["excess"].std() * np.sqrt(ANN_FACTOR)
ranks = sharpes.rank(ascending=False)
start = max(w.first_valid_index() for w in all_trials.values())
bench = backtest.run(
    pd.Series(1.0, index=returns.index).where(returns.index >= start),
    returns,
    lag=LAG,
    cost_bps=COST_BPS,
    rf=rf,
).loc[run["excess"].index]
rows = {
    "buy and hold (benchmark)": stats.summary(
        bench["excess"], total=bench["total"], position=bench["position"]
    )
}

In [9]:
def judge_rule(name):
    """Summary statistics, rank, PSR(0) and DSR for one of your rules."""
    res = run["results"][name]
    m = validation.moments(res["excess"])
    row = stats.summary(res["excess"], total=res["total"], position=res["position"])
    row["rank"] = ranks[name]
    row["PSR(0)"] = validation.psr_from_moments(
        m["sr"], m["n"], m["skew"], m["kurt"], 0.0
    )
    row["DSR"] = validation.dsr(
        res["excess"], n_trials=run["n_ledger"], var_sr=report["var_sr"]
    )
    return row


rows |= {name: judge_rule(name) for name in my_weights}
mine = pd.DataFrame(rows).T[
    [
        "sharpe",
        "ann_mean_excess",
        "max_drawdown",
        "time_in_market",
        "turnover_per_year",
        "rank",
        "PSR(0)",
        "DSR",
    ]
]
display(
    mine.style.format("{:.2f}", na_rep="").format(
        "{:.1%}", subset=["ann_mean_excess", "max_drawdown", "time_in_market"]
    )
)

,sharpe,ann_mean_excess,max_drawdown,time_in_market,turnover_per_year,rank,PSR(0),DSR
buy and hold (benchmark),0.53,9.9%,-55.2%,100.0%,0.03,,,
my_band200,0.66,8.1%,-23.1%,76.7%,1.45,9.00,1.00,0.52


**Reading the table.** On its own the band rule looks convincing: a Sharpe ratio of 0.66 against 0.53 for buy and hold, a worst drawdown of 23% instead of 55%, and a PSR(0) of 1.00. Against the hurdle for 257 tries its DSR is only 0.52, because 0.66 is about what the best of that many useless rules would show. A better idea for the *same* data has to clear an ever higher bar.

## 5. Conclusions

1. **Any idea runs through the referee in one go.** Write a function in Section 3, add it to `MY_RULES`, run all cells. The look ahead check stops a rule that peeks into the future before it is backtested.
2. **The example improves the 200 day rule but does not beat luck.** Less trading and a higher Sharpe ratio (0.66), but DSR 0.52.
3. **The trial count only grows.** 256 after notebook 03, 257 with the example, 258 with your first own rule.

**How to try your own rule:** replace `my_band200` with your function (or add a second one), list it in `MY_RULES`, and choose Run All. To judge it against only your own rules, set `INCLUDE_FAMILY = False`; the ledger still counts the family, because it was tried on this data.

## 6. Appendix: environment

In [10]:
versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "quantlab": version("quantlab"),
}
print(", ".join(f"{k} {v}" for k, v in versions.items()))

python 3.12.11, numpy 2.5.4, pandas 3.0.6, matplotlib 3.11.2, quantlab 0.1.0
